# 🚀 ViLegalNLI Fine-Tuning with BamiBERT (Dual GPU T4 on Kaggle)

- **Mô hình:** `Qualcomm-AI-Research/BamiBERT` (SOTA tiếng Việt encoder, hỗ trợ context tới 2048 tokens).
- **Nhiệm vụ:** Natural Language Inference (NLI) / Legal Verdict Classification (Nhãn `1`: Entailment/Thắng, Nhãn `0`: Contradiction/Thua).
- **Dữ liệu:** Dataset ViLegalNLI (`train.csv` và `val.csv`) từ thư mục `/kaggle/input/datasets/thieuquangvinh/legalnli`.
- **Cấu hình phần cứng:** Kaggle Dual NVIDIA Tesla T4 (2x 16GB VRAM) với PyTorch & Mixed Precision FP16.
- **Chiến lược Truncation:** `max_length = 2048`, thiết lập `truncation="only_second"` để **bảo toàn 100% câu hỏi (`question`)** và chỉ cắt phần đuôi của văn bản án (`context`) nếu tổng token vượt quá 2048.
- **Optimizer & Scheduler:** `AdamW` (`optim="adamw_torch"`, `lr = 2e-5`, `weight_decay = 0.01`) kết hợp **Early Stopping** sau 2 epochs không cải thiện `Macro-F1`.

In [1]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q transformers datasets accelerate evaluate scikit-learn seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [2]:
import os
import sys
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

# Thiết lập seed cố định cho tính tái lập
set_seed(42)

# Kiểm tra cấu hình GPU
print("=== GPU CONFIGURATION ===")
print("CUDA Available:", torch.cuda.is_available())
device_count = torch.cuda.device_count()
print(f"Device Count: {device_count}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | VRAM: {props.total_memory / (1024**3):.2f} GB")

=== GPU CONFIGURATION ===
CUDA Available: True
Device Count: 2
GPU 0: Tesla T4 | VRAM: 14.56 GB
GPU 1: Tesla T4 | VRAM: 14.56 GB


## 📂 2. Tải và chuẩn bị Dữ liệu (ViLegalNLI)

Thư mục Dataset trên Kaggle: `/kaggle/input/datasets/thieuquangvinh/legalnli` (kèm fallback đường dẫn cục bộ).

In [3]:
# Đường dẫn Dataset trên Kaggle
KAGGLE_DATA_DIR = "/kaggle/input/datasets/thieuquangvinh/legalnli"
LOCAL_DATA_DIR = "data"

if os.path.exists(os.path.join(KAGGLE_DATA_DIR, "train.csv")):
    train_path = os.path.join(KAGGLE_DATA_DIR, "train.csv")
    val_path = os.path.join(KAGGLE_DATA_DIR, "val.csv")
elif os.path.exists(os.path.join(LOCAL_DATA_DIR, "train.csv")):
    train_path = os.path.join(LOCAL_DATA_DIR, "train.csv")
    val_path = os.path.join(LOCAL_DATA_DIR, "val.csv")
else:
    raise FileNotFoundError("Không tìm thấy tập dữ liệu train.csv và val.csv!")

print(f"Train path: {train_path}")
print(f"Val path: {val_path}")

df_train = pd.read_csv(train_path, encoding="utf-8")
df_val = pd.read_csv(val_path, encoding="utf-8")

# Kiểm tra và xử lý null / kiểu dữ liệu
df_train = df_train.dropna(subset=["context", "question", "label"]).reset_index(drop=True)
df_val = df_val.dropna(subset=["context", "question", "label"]).reset_index(drop=True)
df_train["label"] = df_train["label"].astype(int)
df_val["label"] = df_val["label"].astype(int)

print(f"\nTrain dataset shape: {df_train.shape}")
print("Train label distribution:")
print(df_train["label"].value_counts())

print(f"\nVal dataset shape: {df_val.shape}")
print("Val label distribution:")
print(df_val["label"].value_counts())

Train path: /kaggle/input/datasets/thieuquangvinh/legalnli/train.csv
Val path: /kaggle/input/datasets/thieuquangvinh/legalnli/val.csv

Train dataset shape: (7705, 3)
Train label distribution:
label
1    4023
0    3682
Name: count, dtype: int64

Val dataset shape: (155, 3)
Val label distribution:
label
1    80
0    75
Name: count, dtype: int64


## ⚙️ 3. Cấu hình Tokenizer & Chiến lược Truncation Context

- Sử dụng Tokenizer của **`Qualcomm-AI-Research/BamiBERT`**.
- Thiết lập **`max_length = 2048`**.
- Khi gọi tokenizer: `text = question`, `text_pair = context` kết hợp tham số **`truncation="only_second"`**:
  - `question` (chuỗi thứ nhất) sẽ luôn được bảo toàn trọn vẹn 100%.
  - `context` (chuỗi thứ hai) sẽ chỉ bị cắt phần đuôi nếu tổng độ dài của cả cặp vượt quá 2048 tokens.

In [4]:
MODEL_NAME = "Qualcomm-AI-Research/BamiBERT"
MAX_TOKENS = 2048

print(f"Loading tokenizer from: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_nli(examples):
    """
    Tokenize cặp (question, context).
    truncation='only_second': Bảo toàn question, chỉ cắt context ở cuối nếu vượt quá 2048 tokens.
    """
    return tokenizer(
        text=examples["question"],
        text_pair=examples["context"],
        max_length=MAX_TOKENS,
        truncation="only_second",
        padding=False # Dynamic padding theo từng batch để tối ưu VRAM
    )

# Chuyển đổi DataFrame sang Hugging Face Dataset
raw_train_ds = Dataset.from_pandas(df_train[["question", "context", "label"]])
raw_val_ds = Dataset.from_pandas(df_val[["question", "context", "label"]])

print("Tokenizing datasets with truncation='only_second' and max_length=2048...")
train_dataset = raw_train_ds.map(preprocess_nli, batched=True, remove_columns=["question", "context"])
val_dataset = raw_val_ds.map(preprocess_nli, batched=True, remove_columns=["question", "context"])

print("\nSample input IDs length:", len(train_dataset[0]["input_ids"]))
print("Decoded token preview:", tokenizer.decode(train_dataset[0]["input_ids"][:80]))

Loading tokenizer from: Qualcomm-AI-Research/BamiBERT


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Tokenizing datasets with truncation='only_second' and max_length=2048...


Map:   0%|          | 0/7705 [00:00<?, ? examples/s]

Map:   0%|          | 0/155 [00:00<?, ? examples/s]


Sample input IDs length: 203
Decoded token preview: <s>Đối với mỗi tội phạm, người phạm tội chỉ bị áp dụng mấy hình phạt chính?</s></s>Các hình phạt đối với người phạm tội
1. Hình phạt chính bao gồm:
a) Cảnh cáo;
b) Phạt tiền;
c) Cải tạo không giam giữ;
d) Trục xuất;
đ) Tù có thời hạn;
e) Tù chung thân;



## 🏗️ 4. Khởi tạo Model & Tối ưu hóa VRAM cho Dual GPU (2x T4)

- `AutoModelForSequenceClassification` với `num_labels = 2`.
- **Gradient Checkpointing** (`gradient_checkpointing_enable()`): Tiết kiệm bộ nhớ GPU cực lớn khi kích thước context lên tới 2048 tokens.
- **DataCollatorWithPadding**: Dynamic batch padding theo chiều dài tối đa của batch hiện tại (`pad_to_multiple_of=8`), tránh lãng phí padding 2048 cố định cho mọi mẫu.

In [5]:
print(f"Loading model {MODEL_NAME} for 2-class NLI Classification...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "CONTRADICTION/LOSE", 1: "ENTAILMENT/WIN"},
    label2id={"CONTRADICTION/LOSE": 0, "ENTAILMENT/WIN": 1}
)

# Kích hoạt Gradient Checkpointing để xử lý context 2048 mượt mà không bị OOM
model.gradient_checkpointing_enable()

# Data Collator dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

Loading model Qualcomm-AI-Research/BamiBERT for 2-class NLI Classification...


model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                        | Status     | 
---------------------------+------------+-
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 📊 5. Hàm Tính Điểm Đánh giá (Metrics)

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

## 🚀 6. Cấu hình TrainingArguments với AdamW & Early Stopping

- **Optimizer:** `AdamW` (`optim="adamw_torch"`, `weight_decay = 0.01`).
- **Learning Rate:** `2e-5` với linear warmup `10%` (`warmup_ratio = 0.1`).
- **Batch Size trên 2 GPU:** `per_device_train_batch_size = 4` với `gradient_accumulation_steps = 2` -> **Effective Batch Size = 4 * 2 GPUs * 2 = 16**.
- **FP16 Mixed Precision:** `fp16 = True` kích hoạt tăng tốc phần cứng Tensor Core trên T4.
- **Early Stopping:** `EarlyStoppingCallback(early_stopping_patience=2)` theo dõi `eval_f1`.

In [ ]:
OUTPUT_DIR = "./bamibert_vilegalnli_output"
BEST_MODEL_DIR = "./bamibert_vilegalnli_best"

# Thu gom bộ nhớ rác trước khi huấn luyện
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    warmup_steps=100,                # Dùng warmup_steps thay cho warmup_ratio bị deprecated
    optim="adamw_torch",             # AdamW Optimizer
    fp16=True,                       # Mixed precision FP16
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=25,
    save_total_limit=1,
    lr_scheduler_type="reduce_lr_on_plateau",
    lr_scheduler_kwargs={"mode": "max", "factor": 0.5, "patience": 3}
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,      # Dùng processing_class thay cho tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("=== BẮT ĐẦU HUẤN LUYỆN BamiBERT ===")
train_result = trainer.train()
print("\n=== KẾT QUẢ HUẤN LUYỆN ===")
print(train_result)


=== BẮT ĐẦU HUẤN LUYỆN BamiBERT ===


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,2.661081,1.249463,0.664516,0.662818,0.664729,0.662917
2,2.453575,0.976230,0.800000,0.795087,0.820166,0.795833
3,2.217672,0.761037,0.845161,0.844064,0.849389,0.843333
4,1.633696,0.613231,0.870968,0.869397,0.881297,0.868333
5,1.354339,0.562961,0.909677,0.909583,0.909583,0.909583
6,1.344555,0.451572,0.922581,0.922577,0.923051,0.923333
7,0.962444,0.443538,0.922581,0.922577,0.924174,0.923750
8,0.754559,0.300994,0.948387,0.948385,0.948867,0.949167
9,0.824548,0.255371,0.967742,0.967720,0.967605,0.967917
10,0.562798,0.258648,0.967742,0.967720,0.967605,0.967917


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


=== KẾT QUẢ HUẤN LUYỆN ===
TrainOutput(global_step=4820, training_loss=1.471757375155247, metrics={'train_runtime': 10471.2968, 'train_samples_per_second': 7.358, 'train_steps_per_second': 0.46, 'total_flos': 3.1195412834388e+16, 'train_loss': 1.471757375155247, 'epoch': 10.0})


## 💾 7. Lưu Best Checkpoint & Đánh giá trên tập Validation

In [8]:
# Lưu mô hình tốt nhất
print(f"Saving best model to {BEST_MODEL_DIR}...")
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print("Lưu model hoàn tất!")

# Đánh giá tổng hợp trên Validation Set
eval_metrics = trainer.evaluate()
print("\n=== METRICS TRÊN TẬP VALIDATION ===")
for k, v in eval_metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

Saving best model to ./bamibert_vilegalnli_best...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Lưu model hoàn tất!


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



=== METRICS TRÊN TẬP VALIDATION ===
eval_loss: 0.2557
eval_accuracy: 0.9677
eval_f1: 0.9677
eval_precision: 0.9676
eval_recall: 0.9679
eval_runtime: 5.0470
eval_samples_per_second: 30.7110
eval_steps_per_second: 3.9630
epoch: 10.0000
